Search Problem Formulation:

->State: The current department the courier robot is located in.

->Initial State: The specific department where the robot begins its delivery (e.g., 'Pharmacy').

->Actions: Moving through an internal corridor to an adjacent, connected department.

->Goal Test: Checking if the robot's current department is equal to the target delivery department.

In [1]:
from collections import deque

# 1. Hospital Graph Representation (8 departments)
hospital_graph = {
    'Pharmacy': ['ER', 'Lab'],
    'ER': ['Pharmacy', 'ICU', 'Radiology'],
    'Lab': ['Pharmacy', 'Radiology', 'Maternity'],
    'Radiology': ['ER', 'Lab', 'Surgery'],
    'ICU': ['ER', 'Surgery', 'Pediatrics'],
    'Surgery': ['Radiology', 'ICU', 'Maternity'],
    'Maternity': ['Lab', 'Surgery', 'Pediatrics'],
    'Pediatrics': ['ICU', 'Maternity']
}

# 3. Breadth-First Search (Shortest Path)
def bfs_route(graph, start, goal):
    # Queue stores tuples of (current_node, path_taken)
    frontier = deque([(start, [start])])
    explored = set()
    nodes_explored_count = 0
    
    while frontier:
        current_node, path = frontier.popleft()
        
        # Skip if already explored to handle cycles
        if current_node in explored:
            continue
            
        explored.add(current_node)
        nodes_explored_count += 1
        
        if current_node == goal:
            return path, nodes_explored_count
            
        for neighbor in graph[current_node]:
            if neighbor not in explored:
                frontier.append((neighbor, path + [neighbor]))
                
    return None, nodes_explored_count

# 4. Depth-First Search (Valid Path, not guaranteed shortest)
def dfs_route(graph, start, goal):
    # Stack stores tuples of (current_node, path_taken)
    frontier = [(start, [start])]
    explored = set()
    nodes_explored_count = 0
    
    while frontier:
        current_node, path = frontier.pop()
        
        if current_node in explored:
            continue
            
        explored.add(current_node)
        nodes_explored_count += 1
        
        if current_node == goal:
            return path, nodes_explored_count
            
        for neighbor in graph[current_node]:
            if neighbor not in explored:
                frontier.append((neighbor, path + [neighbor]))
                
    return None, nodes_explored_count

# 5. Execution and Testing
if __name__ == "__main__":
    test_pairs = [
        ('Pharmacy', 'Surgery'),
        ('Pediatrics', 'Lab')
    ]
    
    for start, goal in test_pairs:
        print(f"\n--- Routing from {start} to {goal} ---")
        
        bfs_path, bfs_nodes = bfs_route(hospital_graph, start, goal)
        bfs_length = len(bfs_path) - 1 # Length is number of corridors (edges)
        print(f"BFS: Path = {' -> '.join(bfs_path)} | Length = {bfs_length} | Nodes Explored = {bfs_nodes}")
        
        dfs_path, dfs_nodes = dfs_route(hospital_graph, start, goal)
        dfs_length = len(dfs_path) - 1
        print(f"DFS: Path = {' -> '.join(dfs_path)} | Length = {dfs_length} | Nodes Explored = {dfs_nodes}")


--- Routing from Pharmacy to Surgery ---
BFS: Path = Pharmacy -> ER -> ICU -> Surgery | Length = 3 | Nodes Explored = 7
DFS: Path = Pharmacy -> Lab -> Maternity -> Pediatrics -> ICU -> Surgery | Length = 5 | Nodes Explored = 6

--- Routing from Pediatrics to Lab ---
BFS: Path = Pediatrics -> Maternity -> Lab | Length = 2 | Nodes Explored = 6
DFS: Path = Pediatrics -> Maternity -> Surgery -> ICU -> ER -> Radiology -> Lab | Length = 6 | Nodes Explored = 7


Analysis and Explanations
For the route from Pharmacy to Surgery, DFS and BFS returned routes of significantly different lengths. BFS found the optimal shortest path of 3 corridors because its queue-based structure searches outward radially, perfectly evaluating all immediate neighbors before moving deeper into the graph. Conversely, DFS found a suboptimal path of 5 corridors because its stack-based structure forced it to blindy commit to a deep, unoptimized branch, wandering sequentially through the Lab, Maternity, Pediatrics, and ICU before finally hitting Surgery.

To support the new 12pm–1pm avoidance rule, the core search algorithms (BFS and DFS) do not need to change at all. Instead, the problem formulation must change by expanding the State space to explicitly include time (e.g., (current_location, current_time)). The successor function would then evaluate the simulated clock and proactively filter out any action transitioning into the restricted department during that one-hour block, meaning the solver will simply never generate paths through it at the wrong time.